# Configuration, Normalizer, and Schema Validation

Interactive validation of real estate configuration, Hinglish unit parsing, and Pydantic search filters.

In [ ]:
# Import configuration loader, normalizer helpers, and filter schema
from src.config import load_domain_config
from src.normalizer import (
    extract_bhk,
    extract_budget,
    normalize_city,
    parse_currency_amount,
)
from src.schema import RealEstateQuery

In [ ]:
# Load real estate domain configuration
config = load_domain_config("real_estate")
aliases = config["city_aliases"]
units = config["units"]
print(f"Domain: {config['domain']}")
print(f"Schema fields: {list(config['schema'].keys())}")

In [ ]:
# Normalize colloquial city names to canonical form
test_cities = ["gurgaon", "bombay", "bangalore", "noida"]
for city in test_cities:
    print(f"{city:<12} -> {normalize_city(city, aliases)}")

In [ ]:
# Convert currency amounts with unit multipliers into INR integers
print(f"25k     -> INR {parse_currency_amount('25', 'k', units):,}")
print(f"1.5 cr  -> INR {parse_currency_amount('1.5', 'cr', units):,}")
print(f"80 lakh -> INR {parse_currency_amount('80', 'lakh', units):,}")

In [ ]:
# Extract budget bounds from Hinglish query patterns
budget_queries = ["1.5 cr tak", "under 25k", "25k to 30k", "starting 50 lakh", "5-8 cr"]
for q in budget_queries:
    b_min, b_max = extract_budget(q, units)
    print(f"{q:<20} -> min={b_min} | max={b_max}")

In [ ]:
# Extract bedroom counts (BHK) from query text
bhk_queries = ["noida 2bhk flat", "3 bhk ready to move", "1 rk studio", "independent house"]
for q in bhk_queries:
    print(f"{q:<24} -> bhk={extract_bhk(q)}")

In [ ]:
# Validate search filter dictionary against Pydantic schema
filter_obj = RealEstateQuery(
    city="Gurugram",
    bhk=3,
    possession="ready_to_move",
    budget_max=15000000,
)
print(filter_obj.model_dump(exclude_none=True))

In [ ]:
# Verify error handling for invalid ranges
try:
    RealEstateQuery(budget_min=20000000, budget_max=10000000)
except ValueError as e:
    print(f"Caught expected range error: {e}")